# M8.3A.1 — Overfitability ladder (spatial readout)

Plan: [`plans/milestone_08/08_single_view_localization_plan.md`](../../plans/milestone_08/08_single_view_localization_plan.md).  
**Not in Final Report** (report Steps 1–4 cover LR + full train/val/test + split sensitivity).  

**Conclusion (Milestone 8 Step 3A.1):** On tiny train subsets, **Fourier and Flatten** drive train error near zero; **pooling** fails despite ~same learned params as Fourier → the bottleneck is **spatial readout**, not parameter count. No generalisation claim here.

Smoke defaults below keep wall time low; raise `NS_LIST` / `NUM_EPOCHS` / `N_REPS` for the full study.


In [1]:
from pathlib import Path
import shutil
import sys
import subprocess

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Could not locate repository root containing pyproject.toml")
    ROOT = ROOT.parent

# Concurrent notebook installs race on ./build (Errno 17 File exists).
for name in ("build", "dist"):
    shutil.rmtree(ROOT / name, ignore_errors=True)
for egg in (ROOT / "src").glob("*.egg-info"):
    shutil.rmtree(egg, ignore_errors=True)

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-cache-dir",
        f"{ROOT}[dl,dev]",
        "-c",
        str(ROOT / "requirements.txt"),
    ]
)

from gummybear.paths import display_path

print(f"ROOT={display_path(ROOT)}")



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
ROOT=.


In [2]:
from tomography_ml_validation.milestone_08 import m8_corpus_paths, describe_paths

# Prefer full high-regime train pool when present; demo is fine for smoke.
DATA_MODE = "full" if (ROOT / "data/generated/m8_1/single_particle").is_dir() else "demo"
paths = m8_corpus_paths(ROOT, data_mode=DATA_MODE)
print(f"DATA_MODE={DATA_MODE}")
print(describe_paths(paths))


DATA_MODE=full
workbook_path=configs/m8/localization_single_particle.xlsx  output_root=data/generated/m8_1/single_particle  cache_root=data/generated/m8_1/single_particle/_cache


In [3]:
from IPython.display import display
import torch
from torch.utils.data import DataLoader

from tomography_ml import get_device
from tomography_ml.localization import (
    build_shared_subsets,
    run_stage1_overfitability,
    summarize_stage1,
    win3e_control_configs,
)
from tomography_ml.studies import M8_CANONICAL_LR_BY_ARCH
from tomography_ml.training import make_batch_xy_single
from tomography_ml.gummybear_data_catalog.task_dataset import CatalogTaskDataset
from tomography_ml_validation.milestone_08 import (
    architecture_grid_dataframe,
    build_m8_localization_dataset,
    default_single_view_task,
)

device = get_device()
print(f"device={device}")
print("canonical LRs", M8_CANONICAL_LR_BY_ARCH)


device=mps
canonical LRs {'pooled': 0.001, 'fourier': 0.03, 'flatten': 0.0003}


## Shared train subsets + mechanism triad


In [4]:
# Smoke ladder: small n, few epochs. Increase for the full memorisation sweep.
NS_LIST = (1, 2, 3)
N_REPS = 1
NUM_EPOCHS = 40
BASE_SEED = 0

task = default_single_view_task(
    keep_angles_deg=180.0,
    optical_setup_id=None if DATA_MODE == "demo" else "opt_m8_high_001",
)
dataset, task = build_m8_localization_dataset(ROOT, data_mode=DATA_MODE, task=task)
assert len(dataset) >= max(NS_LIST), f"Need >= {max(NS_LIST)} train rows; got {len(dataset)}"

configs = win3e_control_configs()  # pool / Fourier / Flatten triad
display(architecture_grid_dataframe(configs))

sequence_ids = [row.sequence_id for row in dataset.rows]
subsets = build_shared_subsets(
    n_pool=len(dataset),
    sequence_ids=sequence_ids,
    ns=NS_LIST,
    n_reps=N_REPS,
    base_seed=BASE_SEED,
    input_representation=task.x_fields[0],
    normalisation=task.image_normalize,
)
print(f"dataset n={len(dataset)}  subsets={len(subsets)}")


,arch_name,head_type,encoder_channels,downsample,maxpool_after_blocks,pool_schedule,pre_flatten_channels,embed_dim,flatten_hidden,flatten_head,input_representation,normalisation
0,fourier_base_mlp,fourier,"(16, 32, 64)",base,none,"3× (Conv3×3 → ReLU), no MaxPool",None,128,128,mlp,anomaly_ref,none
1,flatten_base_mlp,flatten,"(16, 32, 64)",base,none,"3× (Conv3×3 → ReLU), no MaxPool",None,128,128,mlp,anomaly_ref,none
2,pooled_base_base,pooled,"(16, 32, 64)",base,none,"3× (Conv3×3 → ReLU), no MaxPool",None,128,128,mlp,anomaly_ref,none


dataset n=210  subsets=3


In [5]:
x0, _ = dataset[0]
_, _, H, W = x0[task.x_fields[0]].shape
batch_xy = make_batch_xy_single(
    x_field=task.x_fields[0],
    y_fields=task.y_fields,
    device=device,
)

def make_subset_loader(subset):
    subset_ds = CatalogTaskDataset(
        rows=tuple(dataset.rows[i] for i in subset.indices),
        task=task,
    )
    return DataLoader(
        subset_ds,
        batch_size=min(8, len(subset.indices)),
        shuffle=False,
    )

# Architecture-specific LR via train_subset_run is fixed per call; use primary Fourier LR for smoke.
# Full study sweeps per-arch LRs (Final Report Step 1 / M8_CANONICAL_LR_BY_ARCH).
df = run_stage1_overfitability(
    configs=configs,
    subsets=subsets,
    make_subset_loader=make_subset_loader,
    batch_xy=batch_xy,
    n_outputs=3,
    y_fields=task.y_fields,
    device=device,
    sample_hw=(H, W),
    num_epochs=NUM_EPOCHS,
    lr=float(M8_CANONICAL_LR_BY_ARCH.get("fourier", 0.03)),
    experiment_prefix="m8_3a1_smoke",
)
summary = summarize_stage1(df)
display(summary)
print(
    "Interpretation: compare train_RMSE across pooled / fourier / flatten at each n. "
    "Expect pool to lag; Fourier≈Flatten on tiny subsets → spatial readout bottleneck."
)


,architecture_name,parameter_count,n,n_reps,train_loss_final_mean,train_loss_final_std,train_RMSE_total_mean,train_RMSE_total_std,train_RMSE_X_mean,train_RMSE_X_std,train_RMSE_Y_mean,train_RMSE_Y_std,train_RMSE_Z_mean,train_RMSE_Z_std,overfit_success_rate,collapse_rate
0,flatten_base_mlp,134249859,1,1,24.695837,0.0,4.646781,0.0,0.551038,0.0,0.145455,0.0,8.028257,0.0,0.0,0.0
1,flatten_base_mlp,134249859,2,1,8.454216,0.0,2.840503,0.0,3.764090,0.0,3.081142,0.0,0.737268,0.0,0.0,0.0
2,flatten_base_mlp,134249859,3,1,45.135483,0.0,6.683784,0.0,5.812986,0.0,4.179796,0.0,9.097109,0.0,0.0,0.0
3,fourier_base_mlp,40323,1,1,0.189006,0.0,0.350942,0.0,0.334313,0.0,0.501654,0.0,0.077839,0.0,0.0,0.0
4,fourier_base_mlp,40323,2,1,0.118299,0.0,0.382951,0.0,0.205903,0.0,0.520568,0.0,0.355765,0.0,1.0,0.0
5,fourier_base_mlp,40323,3,1,0.083202,0.0,0.378225,0.0,0.298732,0.0,0.336942,0.0,0.475807,0.0,1.0,0.0
6,pooled_base_base,32003,1,1,0.398313,0.0,0.338925,0.0,0.328671,0.0,0.484856,0.0,0.038731,0.0,0.0,0.0
7,pooled_base_base,32003,2,1,7.795241,0.0,2.784211,0.0,3.578486,0.0,3.073442,0.0,1.001939,0.0,0.0,0.0
8,pooled_base_base,32003,3,1,0.891915,0.0,0.676498,0.0,0.787723,0.0,0.268651,0.0,0.824783,0.0,1.0,0.0


Interpretation: compare train_RMSE across pooled / fourier / flatten at each n. Expect pool to lag; Fourier≈Flatten on tiny subsets → spatial readout bottleneck.
